# Crear un usuario

Da de alta a una persona: guarda su nombre y email, y el sistema le asigna un id y la fecha de alta.

**Qué pasa en la base:** ⚠️ **usa la base de datos real** (`src/db/users.sqlite`) y **añade un usuario**. No hay forma de deshacerlo.

**Acción que usa:** `POST http://127.0.0.1:3004/users` con el cuerpo `{"name": ..., "email": ...}`. Responde `201` con el usuario.

| Dato | Significado | ¿Obligatorio? | Ejemplo válido |
|---|---|---|---|
| `name` | Nombre de la persona | Sí, no vacío | `Sofía Martín` |
| `email` | Algo antes y después de `@`, menos de 64 caracteres. Se guarda en minúsculas y no puede repetirse | Sí | `sofia@example.com` |

**Anotaciones:** necesitas `pnpm start` y `pnpm node-http` en marcha. Cambia los dos valores marcados con ✏️ y pulsa **▶ Run** (o `Shift + Enter`). **Cómo reconocer que ha funcionado:** sale una tabla con el usuario nuevo, su `id` y las fechas. Si falla, sale solo el mensaje y ninguna tabla.

- `The name is missing.` (400): el nombre está vacío o solo tiene espacios. Escribe un nombre.
- `The email is not valid.` (400): el email no tiene algo antes y después de la `@`, o tiene 64 caracteres o más. Escríbelo completo.
- `This email is already used.` (409): otro usuario ya tiene ese email (por ejemplo, si repites la prueba sin cambiarlo). Usa un email distinto.
- `The request is not valid.` (400): falta `name` o `email`, o no son texto.
- **Con la base parada:** para el programa de la base de datos (`pnpm start`) y vuelve a pulsar **▶ Run**: sale `The database failed.` y ninguna tabla.
- Si sale `The HTTP client is not running. Start it with pnpm node-http.`, arranca el cliente en otra terminal (con `pnpm start` ya en marcha) y repite.

[Siguiente: Listar usuarios →](list-users.ipynb)

In [ ]:
import json
import os
import subprocess
import urllib.error
import urllib.request

# ✏️ CAMBIA AQUÍ
name = "Sofía Martín"
email = "sofia@example.com"

URL = "http://127.0.0.1:3004/users"
request = urllib.request.Request(
    URL,
    data=json.dumps({"name": name, "email": email}).encode(),
    headers={"Content-Type": "application/json"},
    method="POST",
)

# Envía la petición al cliente. Una respuesta de error (400, 404, 409, 500) también trae un texto
status = None
text = ""
try:
    with urllib.request.urlopen(request) as answer:
        status = answer.status
        text = answer.read().decode()
except urllib.error.HTTPError as error:
    status = error.code
    text = error.read().decode()
except urllib.error.URLError:
    pass

# Código de Node que dibuja las filas en una tabla con cli-table3
js = """
import Table from "cli-table3";

const rows = JSON.parse(process.argv[1]);

const columns = ["id", "name", "email", "createdAt", "updatedAt"];
const table = new Table({ head: columns, style: { head: ["cyan", "bold"] } });

for (const row of rows) {
  table.push(columns.map((column) => row[column]));
}
console.log(table.toString());
"""

if status is None:
    print("The HTTP client is not running. Start it with pnpm node-http.")
elif status >= 400:
    # Un fallo: solo el mensaje, sin tabla
    print(json.loads(text)["error"])
else:
    data = json.loads(text)
    rows = [data]
    # Lanza Node desde la carpeta first-app (tres carpetas más arriba) y le pasa las filas como JSON
    result = subprocess.run(
        ["node", "--disable-warning=ExperimentalWarning", "--input-type=module", "-e", js, json.dumps(rows)],
        cwd="../../..",
        env={**os.environ, "FORCE_COLOR": "1"},  # conserva el color de los encabezados
        capture_output=True,
        text=True,
    )
    print(result.stdout or result.stderr)